# Data Embeddings and AI Modelling for Complaint Operations Intelligence

## Overview
This notebook builds the embedding and downstream AI/ML layer on top of the cleaned complaint dataset produced in the previous data-preparation stage.

The workflow progresses from **text embeddings** to **semantic validation**, **topic discovery**, **business-intent creation**, **supervised classification**, **semantic similarity retrieval**, **new-complaint prediction**, and **model/data persistence**.

## Main objectives
1. Load and validate the AI-ready complaint text.
2. Generate reusable semantic embeddings using `all-MiniLM-L6-v2`.
3. Validate the embedding space using cosine similarity.
4. Explore complaint structure using UMAP and BERTopic.
5. Interpret discovered topics as business-oriented complaint intents.
6. Train and evaluate a Logistic Regression classifier using complaint embeddings.
7. Build a nearest-neighbour similarity component for retrieving similar historical complaints.
8. Demonstrate prediction for a new complaint.
9. Save the enriched dataset, embeddings, classifier, embedding model, and historical complaint data for application use.

## Relationship to the AI Agent
The outputs of this notebook provide several components that can support the later AI Agent/application layer:

- **Embeddings:** numerical semantic representations of complaint text.
- **Business intent classifier:** predicts a business-facing intent for a new complaint.
- **Similarity index:** retrieves semantically similar historical complaints.
- **Historical complaint data:** provides context that can accompany retrieved cases.
- **Saved model artefacts:** allow the application to reuse the trained components without retraining.


## 1. Load the AI-ready complaint dataset
This step loads the cleaned complaint dataset produced by the previous data-preparation stage.

**Purpose:** Use the validated `CleanDataForAI.csv` dataset as the input to the embedding and AI modelling workflow. The `clean_text` field is the main textual representation that will be transformed into numerical embeddings.

In [1]:
import pandas as pd
import numpy as np

ai_df = pd.read_csv("CleanDataForAI.csv", low_memory=False)

## 2. Validate the text input before embedding
This cell performs basic structural checks on the AI-ready dataset and specifically verifies the quality of the `clean_text` field.

**Purpose:** Confirm that the text supplied to the embedding model is present and usable before computationally expensive embedding generation begins.

In [2]:
print(ai_df.shape)
print(ai_df.columns.tolist())
print(ai_df["clean_text"].isna().sum())
print((ai_df["clean_text"].str.strip() == "").sum())

(21071, 15)
['complaint_id', 'complaint_what_happened', 'clean_text', 'product', 'sub_product', 'issue', 'sub_issue', 'date_received', 'date_sent_to_company', 'state', 'submitted_via', 'company_response', 'consumer_disputed', 'tags', 'consumer_consent_provided']
0
0


## 3. Inspect sample complaint text
A random sample of cleaned complaint texts is displayed to provide a qualitative check of the data that will be embedded.

In [3]:
ai_df["clean_text"].sample(5).tolist()

['- sunday, DATE i exited the in new york around . - a gentleman driving a pedicab offered a ride. i asked the price and he said, MONEY per minute. i told him i was going to and asked how much time that should take and what the price would be. he said, that will take about 6 or 7 minutesso it will cost around MONEY i agreed to that price. - when we arrived at the driver said, i am sorry it took a little longer than i quoted you ; it took around 8 and a half minutes. so the charge will be just under MONEY . but here is my business card. if you will call me when your dinner at is through, i will pick you up and take you back to your hotel for free for the discrepancy in price. i was given no invoice and was told it would come through via email. - about an hour later the receipt came via email with a charge of MONEY for the 8 minute ride instead of the agreed-upon amount of under MONEY . my signature is not on the receipt. i immediately pulled out the pedicab drivers card and sent him a t

## 4. Install the sentence-transformers library
This installs or upgrades the `sentence-transformers` package.

**Purpose:** Loads a pretrained Sentence Transformer model and convert complaint text into dense numerical embeddings.

In [4]:
pip install -U sentence-transformers

Note: you may need to restart the kernel to use updated packages.


## 5. Load the embedding model
The pretrained `all-MiniLM-L6-v2` Sentence Transformer model is loaded.

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\HZJC3023\OneDrive - orange.com\Bureau\Me\Task\complaints_flat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


In [6]:
print(embedding_model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


## 6. Generate embeddings for a small sample
Five complaint texts are encoded as a small test batch.

The resulting shape is printed to confirm that the model returns a fixed-length vector for each complaint.

**Purpose:** Perform an initial end-to-end embedding test before processing the complete dataset.

In [7]:
sample_texts = ai_df["clean_text"].head(5).tolist()
print(sample_texts)
sample_embeddings = embedding_model.encode(sample_texts)
print(sample_embeddings.shape)

['good morning my name is and i appreciate it if you could help me put a stop to chase bank cardmember services. in 2018 i wrote to chase asking for debt verification and what they sent me a statement which is not acceptable. i am asking the bank to validate the debt. instead i been receiving mail every month from them attempting to collect a debt. i have a right to know this information as a consumer. chase account # thanks in advance for your help.', 'i upgraded my card in DATE and was told by the agent who did the upgrade my anniversary date would not change. it turned the agent was giving me the wrong information in order to upgrade the account. changed my anniversary date from DATE to DATE without my consent! has the recording of the agent who was misled me.', 'chase card was reported on DATE . however, fraudulent application have been submitted my identity without my consent to fraudulently obtain services. do not extend credit without verifying the identity of the applicant.', '

## 7. Inspect an individual embedding
The first generated embedding vector is displayed.


In [8]:
print(sample_embeddings[0])

[ 8.58079549e-03  4.17677127e-03 -1.31655270e-02 -2.29204874e-02
 -4.68692509e-03 -4.64838520e-02  3.69010121e-02 -5.87931164e-02
 -8.08290765e-02 -8.81320909e-02 -2.91247722e-02 -6.15859404e-02
  4.30019125e-02 -2.24325750e-02 -5.82047775e-02 -2.14126166e-02
  5.18891960e-02  7.77237909e-03  3.34551409e-02  7.83604383e-02
 -2.15428919e-02  1.50959892e-03 -4.86578513e-03  3.30263935e-02
 -8.62231329e-02  9.50293709e-03 -4.72099856e-02 -2.25565527e-02
 -5.71312793e-02 -4.10867892e-02  1.08987264e-01  1.32080838e-02
  1.05771154e-01 -3.45881730e-02  9.75500718e-02 -1.45860552e-03
  2.55398266e-02  5.79842590e-02  3.09832972e-02 -6.62149787e-02
 -3.48065980e-03 -7.22829401e-02 -4.15185206e-02 -4.24611978e-02
  5.11556789e-02  8.50490946e-03 -2.34728288e-02  5.15551120e-02
 -4.75540310e-02  8.73112902e-02 -5.11628985e-02 -8.06042105e-02
 -3.39938849e-02 -9.94643662e-04 -4.06347848e-02  7.33081400e-02
  8.46658349e-02  2.49809772e-02 -7.33950699e-04 -7.65509531e-02
  7.51396045e-02 -2.31626

## 8. Prepare a semantic-similarity test set
Three example complaint sentences are embedded. Two describe a similar cash-deposit problem, while the third describes a mortgage interest-rate issue.

**Purpose:** Create a controlled test case for checking whether the embedding space places semantically similar complaints closer together than unrelated complaints.

In [9]:
test_sentences = [
    "The ATM took my cash deposit but the money was never credited.",
    "I deposited cash at the ATM but the funds never appeared in my account.",
    "My mortgage interest rate increased unexpectedly."
]

test_embeddings = embedding_model.encode(test_sentences)

## 9. Evaluate semantic similarity with cosine similarity
Cosine similarity is calculated between the test complaint embeddings.

**Purpose:** Validate the semantic behaviour of the embeddings. The two cash-deposit statements should have higher similarity to each other than to the unrelated mortgage statement.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(test_embeddings)

print(similarity)

[[1.         0.8638706  0.23002476]
 [0.8638706  1.0000001  0.19402167]
 [0.23002476 0.19402167 1.        ]]


## 10. Prepare all complaint text for embedding
The complete `clean_text` column is converted into a Python list and its size is checked.

In [11]:
texts = ai_df["clean_text"].tolist()
print(len(texts))

21071


## 11. Generate embeddings for the complete dataset
The embedding model encodes every complaint using a batch size of 32.

**Purpose:** Produce the reusable vector representation of the complete complaint corpus.

In [12]:
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

Batches: 100%|██████████| 659/659 [14:06<00:00,  1.28s/it]


In [13]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(21071, 384)


## 12. Persist the initial complaint embeddings
The complete embedding matrix is saved to `complaint_embeddings.npy` to avoid having to regenerate the embeddings every time the notebook is executed.

In [14]:
import numpy as np

np.save("complaint_embeddings.npy", embeddings)

In [15]:
embeddings = np.load("complaint_embeddings.npy")

In [16]:
print(embeddings.shape)

(21071, 384)


In [17]:
embedding_model_name = "all-MiniLM-L6-v2"

## 13. Install topic-modelling dependencies
This installs the libraries used for dimensionality reduction and topic modelling: BERTopic, UMAP, and HDBSCAN.

In [18]:
pip install -U bertopic umap-learn hdbscan

Note: you may need to restart the kernel to use updated packages.


In [19]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

## 14. Reduce embeddings to two dimensions with UMAP
UMAP reduces the high-dimensional complaint embeddings to two dimensions.

In [20]:
from umap import UMAP

umap_model = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

embeddings_2d = umap_model.fit_transform(embeddings)

print(embeddings_2d.shape)

(21071, 2)


## 15. Install Matplotlib for visualization

**Purpose:** Support visual exploration of the UMAP representation.


In [ ]:
pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

plt.scatter(
    embeddings_2d[:, 0],
    embeddings_2d[:, 1],
    s=3,
    alpha=0.5
)

plt.title("Complaint Embeddings Visualized with UMAP")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")

plt.show()

## 16. Visualize embeddings by product
The UMAP points are coloured according to product categories.

**Purpose:** Explore whether complaint embeddings naturally separate according to the existing `product` field.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    embeddings_2d[:, 0],
    embeddings_2d[:, 1],
    c=ai_df["product"].astype("category").cat.codes,
    cmap="tab20",
    s=5,
    alpha=0.5
)

plt.title("Complaint Embeddings by Product")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")

plt.show()

## 17. Install Seaborn for categorical visualization
This installs Seaborn, which is used for a more detailed categorical scatter plot in the next step.

In [ ]:
pip install seaborn

## 18. Visualize embeddings by the most common issues
The ten most frequent complaint issues are selected and displayed on the UMAP representation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the 10 most common issues
top_issues = ai_df["issue"].value_counts().head(10).index

# Create a copy containing only those issues
plot_df = ai_df[ai_df["issue"].isin(top_issues)].copy()

# We need the corresponding UMAP coordinates
plot_df["umap_x"] = embeddings_2d[plot_df.index, 0]
plot_df["umap_y"] = embeddings_2d[plot_df.index, 1]

plt.figure(figsize=(14, 10))

sns.scatterplot(
    data=plot_df,
    x="umap_x",
    y="umap_y",
    hue="issue",
    s=8,
    alpha=0.5,
    palette="tab10"
)

plt.title("Complaint Embeddings by Top 10 Issues")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.show()

In [21]:
pip install bertopic

Note: you may need to restart the kernel to use updated packages.


In [22]:
embeddings.shape

(21071, 384)

In [23]:
len(ai_df)

21071

## 19. Verify complaint-to-embedding alignment
This cell compares the number of complaints, embeddings, and embedding dimensions.

**Purpose:** Confirm that each complaint has a corresponding embedding and that the embedding matrix has a consistent vector dimension.

In [24]:
print("Complaints:", len(ai_df))
print("Embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Complaints: 21071
Embeddings: 21071
Embedding dimensions: 384


## 20. Prepare documents for BERTopic
The cleaned complaint texts are converted into a list named `docs`.

**Purpose:** Provide BERTopic with the original text documents that correspond to the previously generated embeddings.

In [25]:
docs = ai_df["clean_text"].tolist()

In [26]:
print(len(docs))
print(docs[0])

21071
good morning my name is and i appreciate it if you could help me put a stop to chase bank cardmember services. in 2018 i wrote to chase asking for debt verification and what they sent me a statement which is not acceptable. i am asking the bank to validate the debt. instead i been receiving mail every month from them attempting to collect a debt. i have a right to know this information as a consumer. chase account # thanks in advance for your help.


## 21. Configure the initial BERTopic model
A BERTopic model is created with a minimum topic size of 50 complaints.

**Purpose:** Discover recurring themes in the complaint corpus while avoiding extremely small topics.

In [27]:
from bertopic import BERTopic

topic_model = BERTopic(
    min_topic_size=50,
    verbose=True
)

## 22. Run the initial BERTopic model using precomputed embeddings
BERTopic is fitted using both the complaint documents and the previously generated embeddings.

**Purpose:** Discover semantic complaint themes directly from the embedding space.

In [28]:
topics, probabilities = topic_model.fit_transform(
    docs,
    embeddings
)

2026-10-05 23:43:05,311 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-05 23:43:11,576 - BERTopic - Dimensionality - Completed ✓
2026-10-05 23:43:11,578 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-05 23:43:17,804 - BERTopic - Cluster - Completed ✓
2026-10-05 23:43:17,804 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-05 23:43:21,226 - BERTopic - Representation - Completed ✓


## 23. Review the initial topic summary
The BERTopic topic information is displayed.

**Purpose:** Examine topic identifiers, topic sizes, and topic representations to understand the themes discovered by the initial model.

In [29]:
print(topic_model.get_topic_info())

    Topic  Count                                    Name  \
0      -1  10527                        -1_to_the_and_my   
1       0    675              0_fees_overdraft_money_fee   
2       1    672                  1_check_the_bank_funds   
3       2    610               2_dispute_merchant_the_to   
4       3    544               3_points_offer_card_bonus   
5       4    497           4_credit_closed_card_accounts   
6       5    486         5_credit_card_application_chase   
7       6    417            6_payment_late_date_payments   
8       7    339                     7_card_debit_my_atm   
9       8    338      8_interest_balance_statement_money   
10      9    315       9_escrow_insurance_mortgage_taxes   
11     10    308        10_inquiry_credit_inquiries_hard   
12     11    293               11_amazon_card_credit_the   
13     12    273             12_flight_refund_travel_the   
14     13    272                 13_money_tickets_the_to   
15     14    251        14_charges_card_

In [39]:
topic_model.get_topic(0)

[('deposit', np.float64(0.030490884062382664)),
 ('checks', np.float64(0.030364400823307283)),
 ('deposited', np.float64(0.028899515479791756)),
 ('cashed', np.float64(0.017611613133123594)),
 ('branch', np.float64(0.017556350842412045)),
 ('cleared', np.float64(0.016438735403838122)),
 ('check money', np.float64(0.013417261673610357)),
 ('deposited check', np.float64(0.01311012100141337)),
 ('cash', np.float64(0.012292657886536237)),
 ('funds available', np.float64(0.011988365391639279))]

In [ ]:
topic_model.get_topic(1)

## 24. Inspect representative documents for Topic 1
Representative complaint examples for Topic 1 are displayed.

In [ ]:
topic_model.get_representative_docs(1)

## 25. Evaluate topic sizes and outliers
This section examines:
- The number of complaints assigned to each topic.
- The proportion classified as outliers (`Topic = -1`).
- Several selected topics and their representative complaints.

**Purpose:** Assess whether the topic model is producing useful, sufficiently populated groups and how much of the corpus remains unclassified.

In [ ]:
# 1. How many documents are in each topic?
print(topic_model.get_topic_info()[["Topic", "Count", "Name"]])

# 2. What percentage are outliers?
topic_info = topic_model.get_topic_info()

outliers = topic_info.loc[
    topic_info["Topic"] == -1, "Count"
].iloc[0]

total = topic_info["Count"].sum()

print("Outliers:", outliers)
print("Outlier %:", outliers / total * 100)

# 3. Inspect several topics
for topic_id in [0, 1, 2, 3, 4, 5]:
    print("\nTOPIC", topic_id)
    print(topic_model.get_topic(topic_id))
    print(topic_model.get_representative_docs(topic_id)[:2])

In [ ]:
topic_model.get_topic_info()

In [30]:
for topic_id in range(0, 44):
    print(f"\nTOPIC {topic_id}")
    print(topic_model.get_topic(topic_id))
    print(topic_model.get_representative_docs(topic_id)[:2])



TOPIC 0
[('fees', np.float64(0.03495875701790839)), ('overdraft', np.float64(0.033194689517492874)), ('money', np.float64(0.02235334490335673)), ('fee', np.float64(0.020412358260859173)), ('account', np.float64(0.020140971343853193)), ('the', np.float64(0.01625953604369057)), ('charged', np.float64(0.016038978476909188)), ('to', np.float64(0.015708262933770208)), ('my', np.float64(0.014114278072635981)), ('in', np.float64(0.013926391115099059))]
["i have been a chase bank customer for 8 years. over that time, i have paid a high number of overdraft fees and monthly fees to the institution. i feel that these fees are unfair, unethical and a predatory practice by chase bank. i have paid to chase in excess of MONEY in overdraft and monthly to chase fees in 2015 and much more over the course of my account being open. in the past 3 weeks, i have been charged over MONEY in overdraft fees on separate days where funds were unavailable in my account. in these cases, even though i transferred in

## 26. Configure domain-aware stop words and n-gram extraction
A custom `CountVectorizer` is created using English stop words plus domain/company terms such as `chase`, `jpmorgan`, and related variants.

The vectorizer also uses:
- Unigrams and bigrams (`ngram_range=(1, 2)`).
- A minimum document frequency of 5.
- A maximum document frequency of 0.85.

**Purpose:** Improve topic representations by reducing very common domain-specific words that may dominate topic keywords and by allowing meaningful two-word phrases to be considered.

In [31]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer

custom_stopwords = set(ENGLISH_STOP_WORDS).union({
    "chase",
    "jpmorgan",
    "jp",
    "morgan",
    "jpmcb"
})

vectorizer_model = CountVectorizer(
    stop_words=list(custom_stopwords),
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85
)

## 27. Build a refined BERTopic configuration
A new BERTopic model is configured with:
- The Sentence Transformer embedding model.
- The custom vectorizer.
- `min_topic_size=50`.

**Purpose:** Refine topic representation using domain-aware text features while retaining semantic embeddings for the underlying topic model.

In [32]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=50,
    verbose=True
)

In [33]:
topics, probabilities = topic_model.fit_transform(ai_df["clean_text"])

2026-10-05 23:44:13,493 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 659/659 [12:46<00:00,  1.16s/it]
2026-10-05 23:57:00,335 - BERTopic - Embedding - Completed ✓
2026-10-05 23:57:00,337 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-05 23:57:06,324 - BERTopic - Dimensionality - Completed ✓
2026-10-05 23:57:06,326 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-05 23:57:13,940 - BERTopic - Cluster - Completed ✓
2026-10-05 23:57:13,947 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-05 23:57:21,195 - BERTopic - Representation - Completed ✓


In [34]:
for topic_id in range(0, 10):
    print(f"\nTOPIC {topic_id}")
    print(topic_model.get_topic(topic_id))
    print(topic_model.get_representative_docs(topic_id)[:2])


TOPIC 0
[('deposit', np.float64(0.030490884062382664)), ('checks', np.float64(0.030364400823307283)), ('deposited', np.float64(0.028899515479791756)), ('cashed', np.float64(0.017611613133123594)), ('branch', np.float64(0.017556350842412045)), ('cleared', np.float64(0.016438735403838122)), ('check money', np.float64(0.013417261673610357)), ('deposited check', np.float64(0.01311012100141337)), ('cash', np.float64(0.012292657886536237)), ('funds available', np.float64(0.011988365391639279))]
['i was given a check from my father ( who has the same last name as me and has deposited money into my account on multiple occasions, usually at amounts between $ MONEY ). the check amount was for MONEY . it was printed and issued by his bank. i deposited the check on DATE ( saturday ). at the time of the deposit, i was given a receipt stating that the funds would not be available until DATE . i deposited the check in person at a branch. on monday, DATE , i received word from my father that the chec

## 28. Review the refined topic summary
The complete refined topic information is displayed.

**Purpose:** Examine the number and size of discovered topics after refinement.

In [40]:
print(topic_model.get_topic_info())

    Topic  Count                                               Name  \
0      -1  10139                       -1_branch_cards_refund_debit   
1       0    958                  0_deposit_checks_deposited_cashed   
2       1    681  1_overdraft_overdraft fees_insufficient_overdr...   
3       2    655  2_modification_foreclosure_loan modification_p...   
4       3    601             3_branch_account closed_access_savings   
5       4    520                       4_points_bonus_miles_rewards   
6       5    515                      5_merchant_refund_item_vendor   
7       6    463       6_cards_credit cards_identity_identity theft   
8       7    418  7_cards_credit cards_account closed_closed credit   
9       8    329   8_balance transfer_purchases_minimum_promotional   
10      9    305      9_inquiry_inquiries_hard inquiry_inquiry date   
11     10    303                        10_debit_debit card_atm_pin   
12     11    301                      11_amazon_amazon com_com_visa   
13    

## 29. Quantify topic outliers

- Total complaints.
- Number of outliers.
- Percentage of outliers.

**Purpose:** Measure how much of the complaint corpus is not assigned to a discovered topic and provide an objective diagnostic for topic-model quality.

In [41]:
import pandas as pd

topic_series = pd.Series(topics, name="topic")

print("Total complaints:", len(topic_series))
print("Outliers:", (topic_series == -1).sum())
print("Outlier %:", (topic_series == -1).mean() * 100)

Total complaints: 21071
Outliers: 10139
Outlier %: 48.11826681220635


In [42]:
ai_df["topic"] = topics

ai_df["word_count"] = ai_df["clean_text"].str.split().str.len()

print(
    ai_df[ai_df["topic"] == -1]["word_count"].describe()
)

count    10139.000000
mean       251.641582
std        275.267247
min          1.000000
25%         92.000000
50%        180.000000
75%        318.000000
max       4942.000000
Name: word_count, dtype: float64


In [47]:
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [48]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=50,
    verbose=True
)

topics, probabilities = topic_model.fit_transform(
    ai_df["clean_text"]
)

2026-10-06 00:04:13,281 - BERTopic - Embedding - Transforming documents to embeddings.
Batches:   0%|          | 0/659 [00:00<?, ?it/s]

Batches: 100%|██████████| 659/659 [13:06<00:00,  1.19s/it]
2026-10-06 00:17:19,527 - BERTopic - Embedding - Completed ✓
2026-10-06 00:17:19,528 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-06 00:17:36,400 - BERTopic - Dimensionality - Completed ✓
2026-10-06 00:17:36,402 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-06 00:17:43,290 - BERTopic - Cluster - Completed ✓
2026-10-06 00:17:43,296 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-06 00:17:51,330 - BERTopic - Representation - Completed ✓


In [49]:
topic_series = pd.Series(topics)

print("Total:", len(topic_series))
print("Outliers:", (topic_series == -1).sum())
print("Outlier %:", (topic_series == -1).mean() * 100)

print(
    topic_series.value_counts().head(20)
)

Total: 21071
Outliers: 10179
Outlier %: 48.308101181718946
-1     10179
 0       685
 1       679
 2       500
 3       450
 4       431
 5       349
 6       318
 7       298
 8       290
 9       275
 10      274
 11      273
 12      264
 13      263
 14      220
 15      218
 16      210
 17      198
 18      197
Name: count, dtype: int64


## 30. Add product context to the topic-modelling text
A new `topic_text` field combines the product and complaint text into a single model input.

**Purpose:** Give the topic model explicit product context in addition to the complaint wording. This can help distinguish similar complaint language that occurs across different product areas.

In [87]:
ai_df["topic_text"] = (
    "Product: " + ai_df["product"].fillna("") +
    " | Complaint: " + ai_df["clean_text"].fillna("")
)

## 31. Re-run topic modelling using product + complaint context
BERTopic is fitted using the combined `topic_text` representation.

**Purpose:** Discover topics using both the product context and the semantic content of the complaint, supporting more business-oriented topic separation.


In [88]:
topics, probabilities = topic_model.fit_transform(
    ai_df["topic_text"]
)

2026-10-06 01:02:12,106 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 659/659 [13:24<00:00,  1.22s/it]
2026-10-06 01:15:37,045 - BERTopic - Embedding - Completed ✓
2026-10-06 01:15:37,047 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-06 01:15:52,837 - BERTopic - Dimensionality - Completed ✓
2026-10-06 01:15:52,837 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-06 01:15:59,465 - BERTopic - Cluster - Completed ✓
2026-10-06 01:15:59,472 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-06 01:16:07,731 - BERTopic - Representation - Completed ✓


In [90]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3385,-1_prepaid_card prepaid_prepaid card_credit re...,"[prepaid, card prepaid, prepaid card, credit r...","[Product: Credit reporting, credit repair serv..."
1,0,5730,0_savings_savings account_checking savings_acc...,"[savings, savings account, checking savings, a...",[Product: Bank account or service | Complaint:...
2,1,3437,1_mortgage complaint_product mortgage_modifica...,"[mortgage complaint, product mortgage, modific...",[Product: Mortgage | Complaint: the primary re...
3,2,1501,2_merchant_flight_prepaid_prepaid card,"[merchant, flight, prepaid, prepaid card, card...",[Product: Credit card or prepaid card | Compla...
4,3,865,3_product money_money transfer_virtual currenc...,"[product money, money transfer, virtual curren...","[Product: Money transfer, virtual currency, or..."
5,4,667,4_collection_debt collection_collection compla...,"[collection, debt collection, collection compl...",[Product: Debt collection | Complaint: chase b...
6,5,652,5_vehicle_lease_car_vehicle loan,"[vehicle, lease, car, vehicle loan, loan lease...",[Product: Vehicle loan or lease | Complaint: d...
7,6,625,6_bonus_annual_annual fee_miles,"[bonus, annual, annual fee, miles, rewards, pr...",[Product: Credit card or prepaid card | Compla...
8,7,480,7_personal consumer_reports complaint_repair s...,"[personal consumer, reports complaint, repair ...","[Product: Credit reporting, credit repair serv..."
9,8,424,8_personal consumer_reports complaint_services...,"[personal consumer, reports complaint, service...","[Product: Credit reporting, credit repair serv..."


In [91]:
ai_df["topic"] = topics

topic_info = topic_model.get_topic_info()

print(topic_info[["Topic", "Count", "Name"]])

    Topic  Count                                               Name
0      -1   3385  -1_prepaid_card prepaid_prepaid card_credit re...
1       0   5730  0_savings_savings account_checking savings_acc...
2       1   3437  1_mortgage complaint_product mortgage_modifica...
3       2   1501             2_merchant_flight_prepaid_prepaid card
4       3    865  3_product money_money transfer_virtual currenc...
5       4    667  4_collection_debt collection_collection compla...
6       5    652                   5_vehicle_lease_car_vehicle loan
7       6    625                    6_bonus_annual_annual fee_miles
8       7    480  7_personal consumer_reports complaint_repair s...
9       8    424  8_personal consumer_reports complaint_services...
10      9    329  9_purchases_balance transfer_cash advance_advance
11     10    316      10_prepaid card_card prepaid_prepaid_merchant
12     11    298  11_inquiry credit_credit reporting_reports com...
13     12    232                      12_amazon_

## 32. Create a business interpretation of discovered topics
A manual mapping is created from BERTopic topic IDs to business-facing intent labels.

Examples include areas such as mortgages, merchant disputes, debt collection, credit reporting, fraud, interest rates, account closure, and balance transfers.

**Purpose:** Translate machine-discovered topic IDs into meaningful business categories that can be used by downstream complaint operations workflows.

**Important:** These labels are an analyst-defined interpretation of the discovered topics, rather than labels generated directly by BERTopic.

In [92]:
topic_to_business = {
    -1: "Other / Unclassified",
    0: "Bank Account / Savings",
    1: "Mortgage",
    2: "Merchant / Purchase Disputes",
    3: "Money Transfer / Virtual Currency",
    4: "Debt Collection",
    5: "Vehicle Loan / Lease",
    6: "Rewards / Annual Fees / Benefits",
    7: "Credit Reporting",
    8: "Credit Reporting",
    9: "Balance Transfer / Cash Advance",
    10: "Prepaid Card",
    11: "Credit Reporting",
    12: "Amazon / Amazon Credit Card",
    13: "Prepaid Card",
    14: "Prepaid Card",
    15: "Prepaid Card",
    16: "Prepaid Card / Fraud",
    17: "Credit Reporting",
    18: "Bonuses / Promotions / Direct Deposit",
    19: "Student Loan",
    20: "Identity Theft / Fraud",
    21: "Credit Limit / Interest",
    22: "Account Closure",
    23: "APR / Interest Rate",
    24: "Credit Reporting",
    25: "Late Fees / Payment Timing",
    26: "Card / Account Information",
    27: "Credit Reporting",
    28: "Balance Transfer"
}

## 33. Assign business-intent labels to complaints
The topic-to-business mapping is applied to create the `business_intent` field.

A frequency table is then printed.

**Purpose:** Convert unsupervised topic assignments into business-oriented intent labels that can be used as targets for supervised classification.

In [93]:
ai_df["business_intent"] = ai_df["topic"].map(topic_to_business)

print(ai_df["business_intent"].value_counts())

business_intent
Bank Account / Savings                   5730
Mortgage                                 3437
Other / Unclassified                     3385
Credit Reporting                         1546
Merchant / Purchase Disputes             1501
Prepaid Card                              878
Money Transfer / Virtual Currency         865
Debt Collection                           667
Vehicle Loan / Lease                      652
Rewards / Annual Fees / Benefits          625
Balance Transfer / Cash Advance           329
Amazon / Amazon Credit Card               232
Prepaid Card / Fraud                      168
Bonuses / Promotions / Direct Deposit     160
Student Loan                              148
Identity Theft / Fraud                    147
Credit Limit / Interest                   138
Account Closure                           138
APR / Interest Rate                       126
Late Fees / Payment Timing                 71
Card / Account Information                 69
Balance Transfer  

In [95]:
# Keep only records with usable text and a business intent
clf_df = ai_df[
    ai_df["clean_text"].notna() &
    ai_df["business_intent"].notna()
].copy()

# Combine product context with complaint text
clf_df["model_text"] = (
    "Product: " + clf_df["product"].fillna("") +
    " | Complaint: " + clf_df["clean_text"].fillna("")
)

print("Records:", len(clf_df))
print("Classes:", clf_df["business_intent"].nunique())
print(clf_df["business_intent"].value_counts())

Records: 21071
Classes: 22
business_intent
Bank Account / Savings                   5730
Mortgage                                 3437
Other / Unclassified                     3385
Credit Reporting                         1546
Merchant / Purchase Disputes             1501
Prepaid Card                              878
Money Transfer / Virtual Currency         865
Debt Collection                           667
Vehicle Loan / Lease                      652
Rewards / Annual Fees / Benefits          625
Balance Transfer / Cash Advance           329
Amazon / Amazon Credit Card               232
Prepaid Card / Fraud                      168
Bonuses / Promotions / Direct Deposit     160
Student Loan                              148
Identity Theft / Fraud                    147
Credit Limit / Interest                   138
Account Closure                           138
APR / Interest Rate                       126
Late Fees / Payment Timing                 71
Card / Account Information           

## 34. Split data into training and test sets
The labelled dataset is split into 80% training and 20% test data.

Stratification is used so that the class distribution is maintained between the two partitions, and `random_state=42` supports reproducibility.

**Purpose:** Reserve unseen data for an objective evaluation of the business-intent classifier.

In [96]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    clf_df["model_text"],
    clf_df["business_intent"],
    test_size=0.20,
    random_state=42,
    stratify=clf_df["business_intent"]
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("\nTraining class distribution:")
print(y_train.value_counts())

Training records: 16856
Test records: 4215

Training class distribution:
business_intent
Bank Account / Savings                   4584
Mortgage                                 2749
Other / Unclassified                     2708
Credit Reporting                         1237
Merchant / Purchase Disputes             1201
Prepaid Card                              702
Money Transfer / Virtual Currency         692
Debt Collection                           534
Vehicle Loan / Lease                      522
Rewards / Annual Fees / Benefits          500
Balance Transfer / Cash Advance           263
Amazon / Amazon Credit Card               186
Prepaid Card / Fraud                      134
Bonuses / Promotions / Direct Deposit     128
Identity Theft / Fraud                    118
Student Loan                              118
Account Closure                           110
Credit Limit / Interest                   110
APR / Interest Rate                       101
Late Fees / Payment Timing           

## 35. Generate embeddings and train the business-intent classifier
The training and test texts are encoded using the same Sentence Transformer model.

A Logistic Regression classifier is then trained on the training embeddings with:
- `max_iter=1000` to allow sufficient optimization iterations.
- `class_weight="balanced"` to help address class imbalance.
- A fixed random state.

**Purpose:** Learn a supervised mapping from semantic complaint embeddings to business-intent categories.

In [97]:
from sklearn.linear_model import LogisticRegression

# Generate embeddings
X_train_emb = embedding_model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

X_test_emb = embedding_model.encode(
    X_test.tolist(),
    show_progress_bar=True
)

# Train classifier
clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

clf.fit(X_train_emb, y_train)

print("Classifier trained successfully.")

Batches: 100%|██████████| 132/132 [02:23<00:00,  1.09s/it]


Classifier trained successfully.


## 36. Evaluate classifier performance
The trained classifier predicts the test-set business intents.

The notebook reports:
- Accuracy.
- A classification report containing class-level evaluation metrics.

**Purpose:** Assess how well the embedding-based classifier generalizes to previously unseen complaints.

In [98]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = clf.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.8054567022538552

Classification Report:
                                       precision    recall  f1-score   support

                  APR / Interest Rate       0.61      1.00      0.76        25
                      Account Closure       0.29      0.86      0.44        28
          Amazon / Amazon Credit Card       0.67      1.00      0.80        46
                     Balance Transfer       0.46      1.00      0.63        12
      Balance Transfer / Cash Advance       0.68      0.95      0.79        66
               Bank Account / Savings       0.99      0.96      0.97      1146
Bonuses / Promotions / Direct Deposit       0.91      1.00      0.96        32
           Card / Account Information       0.25      0.93      0.39        14
              Credit Limit / Interest       0.46      0.93      0.62        28
                     Credit Reporting       0.76      0.87      0.82       309
                      Debt Collection       0.66      0.93      0.77       13

## 37. Predict business intent for the full labelled dataset
Embeddings are generated for all records in `clf_df`, and the trained classifier predicts a business intent for every record.

**Purpose:** Extend the supervised model from evaluation to operational use across the complaint corpus.

In [99]:
# Predict business intent for all complaints
all_embeddings = embedding_model.encode(
    clf_df["model_text"].tolist(),
    show_progress_bar=True
)

clf_df["predicted_business_intent"] = clf.predict(all_embeddings)

print(clf_df["predicted_business_intent"].value_counts())

Batches: 100%|██████████| 659/659 [12:59<00:00,  1.18s/it]

predicted_business_intent
Bank Account / Savings                   5506
Mortgage                                 3370
Credit Reporting                         1798
Merchant / Purchase Disputes             1797
Prepaid Card                             1181
Debt Collection                           914
Money Transfer / Virtual Currency         898
Rewards / Annual Fees / Benefits          784
Vehicle Loan / Lease                      725
Other / Unclassified                      705
Balance Transfer / Cash Advance           449
Identity Theft / Fraud                    433
Account Closure                           411
Prepaid Card / Fraud                      407
Amazon / Amazon Credit Card               313
Credit Limit / Interest                   277
Card / Account Information                256
Late Fees / Payment Timing                205
Bonuses / Promotions / Direct Deposit     186
APR / Interest Rate                       175
Student Loan                              160
Balance 

## 38. Merge predicted business intent back into the main dataset
The complaint ID and predicted intent are merged back into `ai_df`.

**Purpose:** Make the classifier output available alongside the original complaint attributes so that the enriched dataset can support the application and downstream analytics.

In [100]:
id_map = clf_df[["complaint_id", "predicted_business_intent"]]

ai_df = ai_df.merge(
    id_map,
    on="complaint_id",
    how="left"
)

print(ai_df["predicted_business_intent"].value_counts(dropna=False))

predicted_business_intent
Bank Account / Savings                   5506
Mortgage                                 3370
Credit Reporting                         1798
Merchant / Purchase Disputes             1797
Prepaid Card                             1181
Debt Collection                           914
Money Transfer / Virtual Currency         898
Rewards / Annual Fees / Benefits          784
Vehicle Loan / Lease                      725
Other / Unclassified                      705
Balance Transfer / Cash Advance           449
Identity Theft / Fraud                    433
Account Closure                           411
Prepaid Card / Fraud                      407
Amazon / Amazon Credit Card               313
Credit Limit / Interest                   277
Card / Account Information                256
Late Fees / Payment Timing                205
Bonuses / Promotions / Direct Deposit     186
APR / Interest Rate                       175
Student Loan                              160
Balance 

In [101]:
# Keep the embeddings aligned with clf_df
complaint_embeddings = all_embeddings

print("Embeddings shape:", complaint_embeddings.shape)

Embeddings shape: (21071, 384)


## 39. Persist the operational complaint embeddings
The embeddings are saved to `all_embeddings.npy`.

**Purpose:** Intended to serve as the input to the next stage of the AI pipeline.

In [102]:
import numpy as np

np.save("all_embeddings.npy", complaint_embeddings)

## 40. Test semantic retrieval for one historical complaint
The notebook selects one complaint and retrieves its six nearest neighbours.

For each result, it displays:
- Rank.
- Similarity score derived from cosine distance.
- Business intent.
- Complaint text.

**Purpose:** Validate that the similarity index can retrieve complaints that are semantically close to a selected historical case.


In [103]:
from sklearn.neighbors import NearestNeighbors

similarity_model = NearestNeighbors(
    n_neighbors=10,
    metric="cosine",
    algorithm="brute"
)

similarity_model.fit(complaint_embeddings)

print("Similarity index ready.")

Similarity index ready.


In [104]:
# Pick one complaint
idx = 0

distances, indices = similarity_model.kneighbors(
    complaint_embeddings[idx].reshape(1, -1),
    n_neighbors=6
)

for rank, (neighbor_idx, distance) in enumerate(
    zip(indices[0], distances[0])
):
    print(f"\n--- Rank {rank} | Similarity: {1 - distance:.3f} ---")
    print("Business intent:", clf_df.iloc[neighbor_idx]["predicted_business_intent"])
    print("Complaint:")
    print(clf_df.iloc[neighbor_idx]["clean_text"][:500])


--- Rank 0 | Similarity: 1.000 ---
Business intent: Debt Collection
Complaint:
good morning my name is and i appreciate it if you could help me put a stop to chase bank cardmember services. in 2018 i wrote to chase asking for debt verification and what they sent me a statement which is not acceptable. i am asking the bank to validate the debt. instead i been receiving mail every month from them attempting to collect a debt. i have a right to know this information as a consumer. chase account # thanks in advance for your help.

--- Rank 1 | Similarity: 0.851 ---
Business intent: Debt Collection
Complaint:
i 'm working with and all my creditors know i 'm going through financial difficultly at this time. they all have been informed since DATE . chase credit cards continues to call my work phone even after sending them a letter and telling them not too. i sent chase a certified letter stating to work with who are representing me. they know this and refuse and say it 's not in their policy

In [105]:
for rank, (neighbor_idx, distance) in enumerate(
    zip(indices[0][1:], distances[0][1:]), start=1
):
    row = clf_df.iloc[neighbor_idx]

    print(f"\n--- Rank {rank} | Similarity: {1 - distance:.3f} ---")
    print("Business intent:", row["predicted_business_intent"])
    print("Company response:", row["company_response"])
    print("Complaint:", row["clean_text"][:400])


--- Rank 1 | Similarity: 0.851 ---
Business intent: Debt Collection
Company response: Closed with explanation
Complaint: i 'm working with and all my creditors know i 'm going through financial difficultly at this time. they all have been informed since DATE . chase credit cards continues to call my work phone even after sending them a letter and telling them not too. i sent chase a certified letter stating to work with who are representing me. they know this and refuse and say it 's not in their policy. i ca n't re

--- Rank 2 | Similarity: 0.850 ---
Business intent: Debt Collection
Company response: Closed with explanation
Complaint: on friday, , , i contacted via telephone, to question them about my debt account they purchased from chase bank credit card division. i informed them, that said debt had no activity since , and asked why do they continue to contact me. i was informed that the last payment to chase was . i informed , i have not paid anything on this account since , and t

## 41. Test the end-to-end prediction flow on a new complaint
A new example complaint is combined with its product context and encoded using the same embedding model.

The trained classifier predicts:
- The most likely business intent.
- The top five class probabilities.

**Purpose:** Demonstrate the intended inference workflow for a new complaint entering the system: construct model input → generate embedding → predict business intent → inspect confidence across alternative intents.


In [106]:
test_complaint = (
    "I noticed a cash withdrawal on my account that I don't recognize. "
    "I never made this transaction and Chase has not refunded the money."
)

test_text = (
    "Product: Checking or savings account | "
    "Complaint: " + test_complaint
)

test_embedding = embedding_model.encode([test_text])

predicted = clf.predict(test_embedding)[0]

probs = clf.predict_proba(test_embedding)[0]
classes = clf.classes_

top_idx = probs.argsort()[::-1][:5]

print("Predicted business intent:", predicted)
print("\nTop 5 predictions:")

for i in top_idx:
    print(f"{classes[i]}: {probs[i]:.3f}")

Predicted business intent: Bank Account / Savings

Top 5 predictions:
Bank Account / Savings: 0.940
Bonuses / Promotions / Direct Deposit: 0.037
Money Transfer / Virtual Currency: 0.008
Other / Unclassified: 0.006
Prepaid Card: 0.003


In [107]:
ai_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21071 entries, 0 to 21070
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   complaint_id               21071 non-null  int64
 1   complaint_what_happened    21071 non-null  str  
 2   clean_text                 21071 non-null  str  
 3   product                    21071 non-null  str  
 4   sub_product                18962 non-null  str  
 5   issue                      21071 non-null  str  
 6   sub_issue                  12895 non-null  str  
 7   date_received              21071 non-null  str  
 8   date_sent_to_company       21071 non-null  str  
 9   state                      20928 non-null  str  
 10  submitted_via              21071 non-null  str  
 11  company_response           21071 non-null  str  
 12  consumer_disputed          6261 non-null   str  
 13  tags                       3816 non-null   str  
 14  consumer_consent_provided  21071 

In [108]:
ai_df.to_csv("AI_data_with_classifier.csv",index=False)

## 42. Save the trained models
A `models` directory is created and two key model artefacts are persisted:
- The Logistic Regression business-intent classifier.
- The `all-MiniLM-L6-v2` Sentence Transformer embedding model.

**Purpose:** Make the trained components reusable by the application without retraining them every time the system starts.

In [109]:
import joblib
from pathlib import Path

# Create a models folder
Path("models").mkdir(exist_ok=True)

# Save the classifier
joblib.dump(clf, "models/business_intent_classifier.pkl")

# Save the embedding model
embedding_model.save("models/all-MiniLM-L6-v2")

print("Models saved successfully.")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]

Models saved successfully.


## 43. Persist the historical complaint dataset
A copy of the enriched complaint dataframe is saved as `models/historical_complaints.pkl`.

**Purpose:** Preserve the historical complaint records needed for retrieval and operational use alongside the saved models.


In [110]:
historical_data = ai_df.copy()

historical_data.to_pickle(
    "models/historical_complaints.pkl"
)

print("Historical complaints saved:", historical_data.shape)

Historical complaints saved: (21071, 20)
